# Problem: Write a custom Dataset and Dataloader to load from a CSV file

### Problem Statement
You are tasked with creating a **custom Dataset** and **Dataloader** in PyTorch to load data from a given `data.csv` file. The loaded data will be used to run a pre-implemented linear regression model.

### Requirements
1. **Dataset Class**:
   - Implement a class `CustomDataset` that:
     - Reads data from a provided `data.csv` file.
     - Stores the features (X) and target values (Y) separately.
     - Implements PyTorch's `__len__` and `__getitem__` methods for indexing.

2. **Dataloader**:
   - Use PyTorch's `DataLoader` to create an iterable for batch loading the dataset.
   - Support user-defined batch sizes and shuffling of the data.

In [1]:
import pandas as pd
import torch

torch.manual_seed(42)
X = torch.rand(100, 1) * 10  # 100 data points between 0 and 10
y = 2 * X + 3 + torch.randn(100, 1)  # Linear relationship with noise

# Save the generated data to data.csv
data = torch.cat((X, y), dim=1)
df = pd.DataFrame(data.numpy(), columns=['X', 'y'])
df.to_csv('data.csv', index=False)

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:

import torch.nn as nn
import torch.optim as optim

In [10]:
import torch
from torch.utils.data import Dataset


class LinearRegressionDataset(Dataset):
    def __init__(self, csv_path: str):
        data = pd.read_csv(csv_path)

        self.X = data.iloc[:, :-1].values
        self.y = data.iloc[:, -1].values

    def __len__(self):
        return len(self.X)

    def __getitem__(self, item):
        x = torch.tensor(self.X[item], dtype=torch.float32)
        y = torch.tensor(self.y[item], dtype=torch.float32)
        return x, y


# Example usage of the DataLoader
dataset = LinearRegressionDataset('./data.csv')
dataloader = torch.utils.data.DataLoader(dataset,batch_size=32, shuffle=True)


In [11]:
# Define the Linear Regression Model
class LinearRegressionModel(nn.Module):
    def __init__(self):
        super(LinearRegressionModel, self).__init__()
        self.linear = nn.Linear(1, 1)  # Single input and single output

    def forward(self, x):
        return self.linear(x)


# Initialize the model, loss function, and optimizer
model = LinearRegressionModel()
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# Training loop
epochs = 1000
for epoch in range(epochs):
    for batch_X, batch_y in dataloader:
        # Forward pass
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Log progress every 100 epochs
    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}")


/Users/rezabarzegar/opt/anaconda3/envs/lifting/lib/python3.10/site-packages/torch/nn/modules/loss.py:535: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/Users/rezabarzegar/opt/anaconda3/envs/lifting/lib/python3.10/site-packages/torch/nn/modules/loss.py:535: UserWarning: Using a target size (torch.Size([4])) that is different to the input size (torch.Size([4, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch [100/1000], Loss: 11.7086
Epoch [200/1000], Loss: 67.9492
Epoch [300/1000], Loss: 31.7387
Epoch [400/1000], Loss: 24.3358
Epoch [500/1000], Loss: 59.2137
Epoch [600/1000], Loss: 70.2141
Epoch [700/1000], Loss: 48.2292
Epoch [800/1000], Loss: 27.9624
Epoch [900/1000], Loss: 36.4573
Epoch [1000/1000], Loss: 43.8533


In [12]:
# Display the learned parameters
[w, b] = model.linear.parameters()
print(f"Learned weight: {w.item():.4f}, Learned bias: {b.item():.4f}")

# Testing on new data
X_test = torch.tensor([[4.0], [7.0]])
with torch.no_grad():
    predictions = model(X_test)
    print(f"Predictions for {X_test.tolist()}: {predictions.tolist()}")

Learned weight: 0.6520, Learned bias: 12.5542
Predictions for [[4.0], [7.0]]: [[15.162151336669922], [17.118122100830078]]
